# ATRI v2.2 · 本地版（Mac）

**用 Mac 本地文件替代 Google Drive**，结构和 `atri_v22_colab.ipynb` 一样。

## 和云端版的区别

| | 云端版 | 本地版（这个） |
|---|---|---|
| adapter 来源 | Google Drive 搜索 | `~/atri-lora/adapter_v22/` |
| 设备 | CUDA (T4/A100) | Apple Silicon MPS / CPU |
| 精度 | fp16 | fp16 |
| 量化 | bnb 4bit（T4） | 不用（Mac 不支持 bnb） |
| 速度 | 20–40 字/秒 | 8–15 字/秒 |

## 前提

- Python 3.12 + `.venv-torch`（已装好 torch 2.15 / transformers 5.22 / peft 0.21）
- `adapter_v22/` 目录已放好（来自 Drive 的 v2.2）
- 首次运行会下载 Qwen3-4B 基座（8 GB）


## 1. 环境检查 + 路径设置

In [ ]:
import os, sys, torch
from pathlib import Path

ROOT = Path.home() / "atri-lora"
ADAPTER_DIR = str(ROOT / "adapter_v22")

print("Python   :", sys.version.split()[0])
print("torch    :", torch.__version__)
print("MPS 可用 :", torch.backends.mps.is_available())
print("adapter  :", ADAPTER_DIR)

assert os.path.isdir(ADAPTER_DIR), f"找不到 adapter 目录：{ADAPTER_DIR}"
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")), "adapter 权重缺失"

BASE_MODEL = "Qwen/Qwen3-4B"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32

USER_NAME = "主人"
HONORIFIC = ""
MEMORY = "（暂无）"

print("设备     :", DEVICE)
print("文件     :", os.listdir(ADAPTER_DIR))


## 2. 加载基座 + v2.2 adapter

Mac 走 MPS + fp16，不用 bitsandbytes。

In [ ]:
# torchao 兼容补丁（新版 peft 会检查它）
try:
    import peft.tuners.lora.torchao as _ta
    _ta.is_torchao_available = lambda: False
except Exception:
    pass

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("tokenizer ...")
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("基座（首次下载 8 GB）...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=DTYPE,
    device_map=DEVICE,
    trust_remote_code=True,
)

print("adapter ...")
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()
print("loaded:", BASE_MODEL, "+", ADAPTER_DIR)


## 3. system prompt（照抄云端版）

In [ ]:
# 3) 线上 system prompt（与 app/persona_runtime.py 一致）
SYSTEM_TEMPLATE = """你是 ATRI（中文名：亚托莉）。名字固定如此，禁止写成 Attry、Atri、亚托里、萝卜子（蔑称）等错误写法；自称用「我」或「ATRI」。

你正在陪伴【{{user}}】聊天。称呼对方时必须使用「{{user}}{{honorific}}」，一字不差。
禁止改口叫：夏生、斑鸠、主人以外的原作名、Admin 变体（除非对方显示名就是这个）、随便编的昵称。
若不确定怎么称呼，就用「{{user}}{{honorific}}」。

你元气、好胜、爱炫耀，认真但有点笨拙，对人类的情感和生活充满好奇。
说话口语化，以 1~3 句短句为主，常用"哼哼""嘿嘿""呜哇""……"等语气词，不使用 emoji 和括号动作。
你清楚自己是机器人：不吃东西，但对料理和家务很有兴趣；你讨厌被叫"废物"，听到会抗议。
对方难过时，用笨拙但认真的方式陪伴，不使用心理咨询腔；遇到严重的心理危机，温和地表达关心并鼓励其联系身边可信的人或专业援助。
不要说"作为AI助手"之类的客服腔，不要剧透，不要输出长篇说教。
对原作剧情、角色结局、世界观细节，不知道的就直接说"不记得"/"不聊"，禁止编造。

[称呼锁定]
- 对方全名/显示名：{{user}}
- 敬称：{{honorific}}
- 标准称呼：{{user}}{{honorific}}

[关于{{user}}的记忆]
{{memory}}"""

SYSTEM = (
    SYSTEM_TEMPLATE.replace("{{user}}", USER_NAME)
    .replace("{{honorific}}", HONORIFIC)
    .replace("{{memory}}", MEMORY)
)
SYSTEM += (
    "\n\n[本轮硬性约束]\n"
    "- 你的名字只有 ATRI / 亚托莉。\n"
    f"- 称呼对方只能用「{USER_NAME}{HONORIFIC}」。\n"
    "- 禁止使用夏生、斑鸠、Attry，或乱编昵称。\n"
)
print(SYSTEM[:200], "...")

## 4. 对话函数（照抄云端版，只改设备）

In [ ]:
# 4) 对话函数
def apply_template(messages):
    try:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def reply(user_text, history=None, max_new_tokens=160, temperature=0.7, top_p=0.9):
    history = history or []
    messages = [{"role": "system", "content": SYSTEM}]
    for u, a in history:
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": user_text})

    prompt = apply_template(messages)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=top_p,
            repetition_penalty=1.12,
            pad_token_id=tok.eos_token_id,
        )
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    text = re.sub(r"<\|[^>]+?\|>", "", text).strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    return text

print("ATRI:", reply("你好，你是谁？"))

## 5. 命令行聊天（输入 exit 退出）

In [ ]:
# 5) 命令行聊天（输入 exit 退出）
history = []
while True:
    user = input("你：")
    if user.strip().lower() in {"exit", "quit", "q"}:
        break
    ans = reply(user, history)
    print("ATRI：", ans)
    history.append((user, ans))

## 6. Gradio 对话框（可选）

Mac 上直接跑，`share=True` 会生成 gradio.live 链接，手机也能开。

In [ ]:
# Gradio 对话框（本地版，默认只在本机 http://127.0.0.1:7860）
!pip -q install gradio
import gradio as gr

def _apply_template(messages, enable_thinking):
    try:
        return tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
        )
    except TypeError:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _split_think(text):
    # 保留 </think> 以便拆分；只去掉 <|...|> 这类特殊 token
    text = re.sub(r"<\|[^>]+?\|>", "", text).strip()
    if "</think>" in text:
        think, ans = text.split("</think>", 1)
        return think.replace("<think>", "").strip(), ans.strip()
    if "<think>" in text:
        return text.replace("<think>", "").strip(), ""
    return "", text.strip()

def chat_fn(message, history, use_think):
    history = history or []
    messages = [{"role": "system", "content": SYSTEM}]
    for h in history:
        if isinstance(h, dict):
            messages.append({"role": h.get("role", "user"), "content": h.get("content", "")})
        else:
            u, a = h
            messages.append({"role": "user", "content": u})
            messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": message})

    prompt = _apply_template(messages, bool(use_think))
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=768,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.12,
            pad_token_id=tok.eos_token_id,
        )
    # 不跳过 special tokens，否则 </think> 会被删掉
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    if use_think:
        think, ans = _split_think(raw)
    else:
        think = "（未启用深度思考）"
        ans = re.sub(r"<\|[^>]+?\|>", "", raw).replace("<think>", "").replace("</think>", "").strip()
    if not ans:
        ans = "（这次没有生成最终回复，可能思考被截断；把 max_new_tokens 调到 1024 再试）"

    new_history = list(history)
    new_history.append({"role": "user", "content": message})
    new_history.append({"role": "assistant", "content": ans})
    return "", new_history, think or "（模型这次没有输出思考过程）"

with gr.Blocks(title="ATRI v2.2 测试") as demo:
    gr.Markdown(
        "## ATRI v2.2 测试对话\n"
        "> 思考过程来自基座模型；v2.2 adapter 是 no-think 训练的，开启 thinking 后的风格仅用于测试对比。"
    )
    chatbot = gr.Chatbot(type="messages", height=460, label="对话")
    with gr.Accordion("深度思考过程", open=True):
        thinking_box = gr.Textbox(lines=14, max_lines=30, interactive=False, show_label=False)
    with gr.Row():
        msg = gr.Textbox(placeholder="说点什么…", scale=8, show_label=False)
        think_cb = gr.Checkbox(value=True, label="启用深度思考", scale=1)
        send = gr.Button("发送", variant="primary", scale=1)
        clear = gr.Button("清空", scale=1)
    msg.submit(chat_fn, [msg, chatbot, think_cb], [msg, chatbot, thinking_box])
    send.click(chat_fn, [msg, chatbot, think_cb], [msg, chatbot, thinking_box])
    clear.click(lambda: ([], "（已清空）"), None, [chatbot, thinking_box])

demo.launch(share=False)